# Google Colab GPU OCR Benchmarking Experiment

This notebook runs a performance experiment comparing PaddleOCR on Google Colab's GPU against the CPU baseline (~219,540 ms) from your local environment.

### **Instructions:**
1. **Enable GPU runtime:** Go to **Runtime** > **Change runtime type** in the top menu, select a **GPU** (T4, L4, or any available GPU) as the Hardware Accelerator, and click **Save**.
2. **Run Cells:** Run each cell in order.
3. **Upload Files:** When you run the **Upload Files** cell, click the "Choose Files" button to upload `sample_document.png` and `land_document_extractor.py` from your local project directory.
4. **Report Results:** Copy the printed output from the bottom benchmarking cells and paste it back into your chat conversation.

In [ ]:
# Cell 1: Environment & Hardware Detection
import sys
import subprocess

print(f"Python Version: {sys.version}")

print("\n=== Running nvidia-smi ===")
try:
    print(subprocess.check_output(["nvidia-smi"]).decode("utf-8"))
except Exception as e:
    print(f"NVIDIA-SMI failed: {e}. Please ensure GPU accelerator is selected in Runtime > Change runtime type.")

print("\n=== Checking CUDA Compiler ===")
try:
    print(subprocess.check_output(["nvcc", "--version"]).decode("utf-8"))
except Exception as e:
    print(f"nvcc failed: {e}")

In [ ]:
# Cell 2: Install paddlepaddle-gpu and paddleocr with real-time progress streaming
import os
import re
import sys
import subprocess

def detect_cuda_version():
    try:
        nvcc_out = subprocess.check_output(["nvcc", "--version"]).decode("utf-8")
        match = re.search(r"release (\d+\.\d+)", nvcc_out)
        if match:
            return match.group(1)
    except Exception:
        pass
    try:
        smi_out = subprocess.check_output(["nvidia-smi"]).decode("utf-8")
        match = re.search(r"CUDA Version:\s*(\d+\.\d+)", smi_out)
        if match:
            return match.group(1)
    except Exception:
        pass
    return None

cuda_version = detect_cuda_version()
print(f"Detected CUDA version: {cuda_version}")

# Determine best-matching cuNN/CUDA index for mirror
cu_suffix = "cu120"
if cuda_version:
    try:
        parts = cuda_version.split(".")
        major = int(parts[0])
        minor = int(parts[1]) if len(parts) > 1 else 0
        if major == 11:
            cu_suffix = "cu118"
        elif major == 12:
            if minor >= 6:
                cu_suffix = "cu126"
            elif minor >= 2:
                cu_suffix = "cu122"
            else:
                cu_suffix = "cu120"
        else:
            cu_suffix = f"cu{major}{minor}"
    except Exception:
        pass

print(f"Installing PaddlePaddle-GPU for {cu_suffix}...")
install_cmd = f"pip install paddlepaddle-gpu -i https://www.paddlepaddle.org.cn/packages/stable/{cu_suffix}/"
print(f"Running command: {install_cmd}")

# get_ipython().system streams the output interactively so we see progress
from IPython import get_ipython
ipython = get_ipython()
ipython.system(install_cmd)

print("\nInstalling PaddleOCR 3.7.0...")
ipython.system("pip install paddleocr==3.7.0")

In [ ]:
# Cell 3: Verify GPU compiling in Paddle
import subprocess
import paddle

print(f"Paddle Version: {paddle.__version__}")
print(f"Is Paddle compiled with CUDA: {paddle.is_compiled_with_cuda()}")
print(f"Device in use: {paddle.device.get_device()}")

print("\nRunning a tiny tensor calculation on GPU...")
try:
    x = paddle.to_tensor([10.0, 20.0, 30.0])
    y = paddle.to_tensor([1.0, 2.0, 3.0])
    z = x * y
    print(f"Tensor calculation z = x * y: {z.numpy()}")
except Exception as e:
    print(f"Tensor calculation failed: {e}")

print("\n=== GPU Status after initialization ===")
try:
    print(subprocess.check_output(["nvidia-smi"]).decode("utf-8"))
except Exception:
    pass

In [ ]:
# Cell 4: Upload Required Files
from google.colab import files
import os

print("Please click below to choose and upload 'sample_document.png' and 'land_document_extractor.py' from your local workspace:")
uploaded = files.upload()

print("\n=== Verification of Uploaded Files ===")
for filename in ["sample_document.png", "land_document_extractor.py"]:
    if os.path.exists(filename):
        print(f"✓ '{filename}' is present in the workspace.")
    else:
        print(f"✗ ERROR: '{filename}' was NOT found. Please re-run this cell and upload it.")

In [ ]:
# Cell 5: Initialize PaddleOCR and Benchmark OCR Inference Speed
import time
import cv2
from paddleocr import PaddleOCR

print("Initializing PaddleOCR engine with GPU... (using exact project parameters)")
ocr = PaddleOCR(
    device='gpu',
    lang='en',
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=False
)

img_path = "sample_document.png"
image = cv2.imread(img_path)
if image is None:
    raise ValueError(f"Unable to read image: {img_path}")

print("\n--- Starting Benchmark Inferences ---")
# 1. One Warm-up Run
print("Running warm-up inference (compiling model, loading parameters into GPU memory)...")
w_start = time.perf_counter()
ocr.predict(image)[0]
w_time = (time.perf_counter() - w_start) * 1000
print(f"Warm-up finished in {w_time:.2f} ms\n")

# 2. Three Timed Runs
runs = []
result = None
for i in range(3):
    print(f"Running timed OCR inference {i+1}...")
    start = time.perf_counter()
    result = ocr.predict(image)[0]
    elapsed = (time.perf_counter() - start) * 1000
    runs.append(elapsed)
    print(f"Run {i+1} time: {elapsed:.2f} ms")

avg_time = sum(runs) / 3
cpu_baseline = 219540.0
speedup = cpu_baseline / avg_time

print("\n================ BENCHMARK RESULTS ================")
print(f"GPU Run 1: {runs[0]:.2f} ms")
print(f"GPU Run 2: {runs[1]:.2f} ms")
print(f"GPU Run 3: {runs[2]:.2f} ms")
print(f"GPU Average: {avg_time:.2f} ms")
print(f"CPU Baseline: {cpu_baseline:.2f} ms")
print(f"Estimated Speedup: {speedup:.2f}x")
print("===================================================")

In [ ]:
# Cell 6: Show OCR Text Output and Run Existing Extraction Parser
import json
from land_document_extractor import OCRLine, group_words_into_lines, OCRWord, normalize_space, extract_land_document_from_lines

print("=== First 20 Lines of Detected OCR Text ===")
if result and "rec_texts" in result:
    for idx, text in enumerate(result["rec_texts"]):
        if idx < 20:
            print(f"{idx+1}: {text}")
    if len(result["rec_texts"]) > 20:
        print(f"... and {len(result['rec_texts']) - 20} more lines.")
else:
    print("No OCR lines detected.")

print("\n=== Running Extraction Parser on GPU OCR Results ===")
try:
    # Translate PaddleOCR 3.x result dictionary into OCRLine format expected by the extractor
    words = []
    for text, score, poly in zip(result["rec_texts"], result["rec_scores"], result["rec_polys"]):
        cleaned = normalize_space(str(text))
        if not cleaned:
            continue
        words.append(
            OCRWord(
                text=cleaned,
                score=float(score),
                points=[[int(point[0]), int(point[1])] for point in poly.tolist()],
            )
        )
        
    lines = group_words_into_lines(words)
    raw_text = "\n".join(l.text for l in lines)
    extracted_data = extract_land_document_from_lines(lines, raw_text, "sample_document.png")
    
    # Print the validation values clearly
    print("\n--- Structured Extraction Validation Data ---")
    print(f"Document number:      {extracted_data.get('document_number')}")
    print(f"S.C. / Serial number: {extracted_data.get('serial_number')}")
    print(f"Stamp number:         {extracted_data.get('stamp_information', {}).get('stamp_number')}")
    print(f"Stamp value:          {extracted_data.get('stamp_information', {}).get('stamp_value')}")
    print(f"Document date:        {extracted_data.get('document_date')}")
    print(f"Execution date:       {extracted_data.get('execution_date')}")
    print("Parties:")
    for idx, party in enumerate(extracted_data.get('parties', [])):
        print(f"  {idx+1}. {party.get('name')} - {party.get('role')}")
    print(f"Stamp vendor:         {extracted_data.get('stamp_information', {}).get('stamp_vendor')}")
    print(f"Signature detected:   {extracted_data.get('document_features', {}).get('signature_detected')}")
    print(f"Property info status: {extracted_data.get('property', {}).get('status')}")
    
except Exception as e:
    print(f"Failed to run extraction parser: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
# Cell 7: Expose Colab OCR Engine as a Temporary HTTP API for the Website

import time
import cv2
import numpy as np
import subprocess
import threading
from flask import Flask, request, jsonify

# 1. Detect GPU model name
gpu_name = "NVIDIA GPU"
try:
    smi_out = subprocess.check_output(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"]).decode("utf-8")
    gpu_name = smi_out.strip()
except Exception:
    pass

print(f"Detected GPU: {gpu_name}")

app = Flask(__name__)

@app.route("/", methods=["GET"])
@app.route("/status", methods=["GET"])
def status_endpoint():
    return jsonify({
        "status": "connected",
        "gpu_name": gpu_name
    })

@app.route("/ocr", methods=["POST"])
def ocr_endpoint():
    if "image" not in request.files:
        return jsonify({"error": "No image file provided"}), 400
    
    file = request.files["image"]
    img_bytes = file.read()
    nparr = np.frombuffer(img_bytes, np.uint8)
    image = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
    
    if image is None:
        return jsonify({"error": "Failed to decode image"}), 400
        
    start_time = time.perf_counter()
    # run predict on the gpu ocr model initialized in Cell 5
    result = ocr.predict(image)[0]
    ocr_time = (time.perf_counter() - start_time) * 1000
    
    # Convert numpy arrays to list for JSON serialization
    rec_polys_list = [poly.tolist() for poly in result["rec_polys"]]
    
    return jsonify({
        "rec_texts": result["rec_texts"],
        "rec_scores": [float(s) for s in result["rec_scores"]],
        "rec_polys": rec_polys_list,
        "ocr_time_ms": ocr_time,
        "gpu_name": gpu_name
    })

def run_flask():
    # Run flask on port 5000
    app.run(host="0.0.0.0", port=5000, debug=False, use_reloader=False)

# Start Flask in a background daemon thread
threading.Thread(target=run_flask, daemon=True).start()
time.sleep(2)
print("Flask server successfully started locally on port 5000!")

# Expose the API to the internet using localtunnel or ngrok
print("\n=== Exposing Colab Port 5000 ===")
print("Method A (Localtunnel): Click the URL printed below.")
# Get the public IP of this Colab machine (sometimes localtunnel asks for it as a security verification code)
try:
    public_ip = subprocess.check_output(["curl", "ifconfig.me"]).decode("utf-8").strip()
    print(f"If prompted by localtunnel, enter this Endpoint IP: {public_ip}")
except Exception:
    pass

get_ipython().system("npx localtunnel --port 5000")